# Chapter 04-04 · Splitting II: grouped and chronological

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** moderate - the ideas are simple and the
consequences are not

**Prerequisites:** 04-03 for held-out data and the split lottery, 04-01 for the wall and E18's
member-month table.

**Position in the learning path:** module 04, chapter 4 of 8.

---

## Why this matters

04-03 followed every rule correctly and ended by admitting that all of them rest on one assumption it
never checked: **that one row is one independent observation.**

For the table this course has been using, that is false. 04-01's E18 built the member-month version of
the churn problem - 6,318 rows in which the average member appears about eleven times - and a random
split scatters one person's eleven rows across both sides. The model then meets, at "test" time, people
it has already studied.

This chapter measures what that costs. The answer has a shape worth knowing in advance:

- for a **logistic regression on smooth features** it costs almost nothing - **+0.0021** of AUC
- for a **random forest** it costs **+0.0957**
- for **k-nearest-neighbours** it costs **+0.1052**

and, more seriously than any of those, **the random split picks the wrong model.** It ranks the forest
above the logistic regression. An honest split ranks them the other way round. The mistake is not that a
number is too high; it is that you ship the wrong thing.

## What you will be able to do

- Recognise when rows are not independent, from the data rather than from being told
- Draw the difference between a random and a grouped split, and explain it from the picture
- Measure split inflation yourself, and predict which models will suffer from it
- Split by time when time matters, and say why a random split is time travel
- Use `GroupKFold` and forward-chaining validation, and know which question each answers
- Choose a splitting strategy from two questions about the data

## Warm-up: retrieve, do not reread

1. What does the test set measure that the validation set cannot?
2. What does stratification fix, and what does it leave alone?
3. In 04-01, why could `months_on_file` not be used as a feature?

<br>

*Answers: (1) the performance of the model you chose, because nothing about the choice used it. (2) the
class balance of the split, exactly; not which rows land where. (3) its value was only knowable after the
cancellation it was predicting had happened.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# SYNTHETIC. The gym panel from 04-01.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])


# 04-01's E18: build the problem at every month, then stack. Features from history only.
def frame_at(cut, horizon=6):
    active = panel[(panel.month == cut) & (panel.cancelled == 0)].member_id.unique()
    if len(active) == 0:
        return None
    history = panel[panel.member_id.isin(active) & (panel.month <= cut)]
    future = panel[panel.member_id.isin(active) & (panel.month > cut)]
    left = future[(future.month <= cut + horizon) & (future.cancelled == 1)].member_id.unique()
    by_member = history.groupby("member_id")
    frame = pd.DataFrame({
        "visits_now": history[history.month == cut].set_index("member_id").visits.reindex(active),
        "mean_visits_last_3": history[history.month > cut - 3].groupby("member_id").visits.mean().reindex(active),
        "lifetime_mean_visits": by_member.visits.mean().reindex(active),
        "lifetime_sd_visits": by_member.visits.std().reindex(active).fillna(0),
        "join_month": by_member.month.min().reindex(active),
        "tenure_months": by_member.size().reindex(active),
        "tickets_so_far": by_member.tickets.sum().reindex(active),
    })
    frame["member_id"] = active
    frame["cut_month"] = cut
    frame["target"] = np.isin(active, left).astype(int)
    return frame.reset_index(drop=True)


stacked = pd.concat([frame_at(cut) for cut in range(1, 19)], ignore_index=True)
FEATURES = ["visits_now", "mean_visits_last_3", "lifetime_mean_visits", "lifetime_sd_visits",
            "join_month", "tenure_months", "tickets_so_far"]
X, y, groups = stacked[FEATURES], stacked.target, stacked.member_id

print("%d rows, %d members, %.2f%% cancel within the horizon"
      % (len(stacked), stacked.member_id.nunique(), 100 * y.mean()))
print("the same member appears %.1f times on average, up to %d"
      % (len(stacked) / stacked.member_id.nunique(), stacked.member_id.value_counts().max()))

## The first question: are the rows independent?

You can answer it without any theory. **Count how often the same entity appears.**

In [ ]:
appearances = stacked.member_id.value_counts()
print("rows per member: min %d, median %.0f, max %d"
      % (appearances.min(), appearances.median(), appearances.max()))
print()
one_member = stacked[stacked.member_id == appearances.index[0]].head(4)
print("four consecutive rows for member %d:" % appearances.index[0])
print(one_member[["cut_month", "lifetime_mean_visits", "join_month", "tenure_months", "target"]]
      .to_string(index=False))

**They are almost the same row.** `join_month` is identical by definition, `lifetime_mean_visits` barely
moves, `tenure_months` increases by exactly one each time, and the target is often the same value in
consecutive months because it asks about overlapping windows.

That is what "not independent" means in practice: **knowing one row tells you most of another.** A model
that has seen months 7, 8 and 9 for a member does not need to generalise to predict their month 10 - it
needs to remember.

## What a random split does to that

Here is the whole chapter in one picture. Ten members, their rows laid out along the months, coloured by
which side of the split they land on.

In [ ]:
def draw_split(ax, assignment, title, sample):
    for row, member in enumerate(sample):
        rows_for_member = stacked[stacked.member_id == member]
        for _, record in rows_for_member.iterrows():
            side = assignment[record.name]
            ax.add_patch(plt.Rectangle((record.cut_month - 0.42, row - 0.38), 0.84, 0.76,
                                       facecolor="#0072B2" if side == 0 else "#D55E00",
                                       edgecolor="white", linewidth=0.8))
    ax.set_xlim(0.3, 18.7)
    ax.set_ylim(-0.7, len(sample) - 0.3)
    ax.set_yticks(range(len(sample)))
    ax.set_yticklabels(["member %d" % m for m in sample], fontsize=8)
    ax.set_xlabel("month the prediction is made")
    ax.set_title(title, fontsize=11)


busiest = stacked.member_id.value_counts().index[:10]
sample_rows = stacked[stacked.member_id.isin(busiest)]

rng = np.random.default_rng(0)
random_side = pd.Series(rng.integers(0, 2, len(stacked)), index=stacked.index)

grouped_side = stacked.member_id.map(
    dict(zip(stacked.member_id.unique(), rng.integers(0, 2, stacked.member_id.nunique()))))

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
draw_split(left, random_side, "A random split: every member is on both sides", busiest)
draw_split(right, grouped_side, "A grouped split: each member is on one side only", busiest)
fig.legend(handles=[Patch(facecolor="#0072B2", label="training"),
                    Patch(facecolor="#D55E00", label="test")],
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.show()

**On the left, every single member has rows in both colours.** The model is trained on member 314's
months 3 to 9, then tested on their month 10 - a person whose habits, tenure and lifetime average it has
already memorised.

On the right, a member is entirely one colour. The test rows belong to people the model has never seen,
which is what the production question actually asks: *a new member walks in - what do we predict?*

Put a number on the left-hand picture.

In [ ]:
from sklearn.model_selection import train_test_split

train_index, test_index = train_test_split(np.arange(len(stacked)), test_size=0.3, random_state=0)
train_members = set(stacked.member_id.iloc[train_index])
test_members = set(stacked.member_id.iloc[test_index])
seen_before = len(train_members & test_members)

print("under a random 70/30 split of rows:")
print("  members with at least one test row      : %d" % len(test_members))
print("  of those, also present in training      : %d  (%.1f%%)"
      % (seen_before, 100 * seen_before / len(test_members)))
print("  genuinely unseen members in the test set: %d" % (len(test_members) - seen_before))

fig, ax = plt.subplots(figsize=(8.5, 1.9))
ax.barh([0], [seen_before], color="#D55E00", label="already seen in training")
ax.barh([0], [len(test_members) - seen_before], left=[seen_before], color="#0072B2",
        label="genuinely new")
ax.text(seen_before / 2, 0, "%d members (%.1f%%)" % (seen_before, 100 * seen_before / len(test_members)),
        ha="center", va="center", color="white", fontsize=10)
ax.set_yticks([])
ax.set_xlabel("members appearing in the test set")
ax.set_title("What the 'held-out' set is actually made of", fontsize=11)
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

**539 of 544 test members - 99.1% - were in the training set.** Five members out of 580 are genuinely
held out. The test set is not a sample of new members; it is a sample of new *months* for members the
model already knows.

Whether that matters depends entirely on the next question.

## How much does it cost? It depends on the model

**Predict before running:** three models - a logistic regression, a random forest, and
k-nearest-neighbours. Which will be most inflated by a random split, and why?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def build(name):
    if name == "random forest":
        return RandomForestClassifier(n_estimators=300, random_state=0, min_samples_leaf=1)
    if name == "k-nearest neighbours":
        return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))


def cross_validate(name, splitter, **split_args):
    folds = []
    for train_rows, test_rows in splitter.split(X, y, **split_args):
        model = build(name).fit(X.iloc[train_rows], y.iloc[train_rows])
        folds.append(roc_auc_score(y.iloc[test_rows],
                                   model.predict_proba(X.iloc[test_rows])[:, 1]))
    return np.mean(folds), np.std(folds)


rows = []
for name in ["logistic regression", "random forest", "k-nearest neighbours"]:
    random_mean, random_sd = cross_validate(name, StratifiedKFold(5, shuffle=True, random_state=0))
    grouped_mean, grouped_sd = cross_validate(name, GroupKFold(5), groups=groups)
    rows.append({"model": name,
                 "random row split": round(random_mean, 4),
                 "grouped by member": round(grouped_mean, 4),
                 "inflation": round(random_mean - grouped_mean, 4)})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

**The logistic regression is barely affected (+0.0021). The forest and the neighbours are inflated by
about a tenth of an AUC.**

The mechanism is exactly what you would guess once stated: **inflation is proportional to a model's
capacity to memorise.**

- **Logistic regression** fits seven coefficients. It has nowhere to store a fact about member 314; all
  it can express is a smooth trend across everybody. Seeing that member's other months barely helps it.
- **k-nearest neighbours** *is* memory. At test time it looks for the five most similar rows in the
  training set - and for member 314's month 10, the five most similar rows in the whole dataset are
  member 314's months 7, 8, 9, 11 and 12. It is not predicting; it is doing a lookup on a person it has
  already been given the answer for.
- **A random forest with unlimited leaves** sits between the two and behaves much closer to the
  neighbours, because a deep tree can carve out a leaf per member.

**So a random split does not inflate scores uniformly - it inflates the flexible models most.** And that
is the part that hurts.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.8))

position = np.arange(len(comparison))
left.barh(position + 0.19, comparison["random row split"], height=0.36,
          color="#D55E00", label="random row split")
left.barh(position - 0.19, comparison["grouped by member"], height=0.36,
          color="#0072B2", label="grouped by member")
for index, record in comparison.iterrows():
    left.annotate("", xy=(record["random row split"], index + 0.19),
                  xytext=(record["grouped by member"], index + 0.19),
                  arrowprops=dict(arrowstyle="-", color="#000000", linewidth=0.8))
    left.text(record["random row split"] + 0.006, index + 0.19,
              "+%.4f" % record["inflation"], va="center", fontsize=9)
left.set_yticks(position)
left.set_yticklabels(comparison.model.str.replace(" ", "\n"), fontsize=8)
left.set_xlim(0.5, 0.83)
left.set_xlabel("AUC")
left.set_title("The flexible models are the ones that get inflated", fontsize=11)
left.legend(fontsize=8, loc="lower right")

for index, record in comparison.iterrows():
    colour = ["#0072B2", "#D55E00", "#009E73"][index]
    right.plot([0, 1], [record["random row split"], record["grouped by member"]],
               "o-", color=colour, linewidth=2.2, markersize=8, label=record.model)
    right.text(-0.04, record["random row split"], "%.4f" % record["random row split"],
               ha="right", va="center", fontsize=9, color=colour)
    right.text(1.04, record["grouped by member"], "%.4f" % record["grouped by member"],
               ha="left", va="center", fontsize=9, color=colour)
right.set_xlim(-0.45, 1.45)
right.set_xticks([0, 1])
right.set_xticklabels(["random split\n(what you would have believed)",
                       "grouped split\n(what is true)"], fontsize=9)
right.set_ylabel("AUC")
right.set_title("The lines cross: the split chose the model", fontsize=11)
right.legend(fontsize=8, loc="lower left")

plt.tight_layout()
plt.show()

## The right-hand panel is the real damage

Read it as a ranking rather than as two numbers.

**Under the random split** the random forest is the best model - 0.7580 against the logistic
regression's 0.7313. You would ship the forest, and you would be able to justify it: it was
cross-validated, on held-out folds, by a five-fold procedure, and it won.

**Under the grouped split** the ordering reverses. The logistic regression scores 0.7292 and the forest
manages 0.6624. On members it has never met, the simple model is better by a wide margin.

**The lines cross.** That is worth stating as plainly as possible:

> A leaky split does not merely make your number too big. **It systematically favours the models that
> leak best**, so it changes which model you choose - and the ones it favours are exactly the flexible
> ones that had the least to offer.

An inflated score alone would be survivable; you would be disappointed in production and would recalibrate.
Choosing the wrong model is not survivable in the same way, because the mistake is baked into the
artefact you shipped, and every future comparison is made against the wrong incumbent.

Note also which model was *least* damaged. **The logistic regression's honest score, 0.7292, is very
close to its inflated one, 0.7313** - so a modeller who had only ever fitted a logistic regression would
have got away with a random split entirely. That is why this bug survives: it is invisible until somebody
introduces a model flexible enough to exploit it, and then it silently rewards that model.

## The second question: does time matter?

Grouping fixes *who* is on each side. It says nothing about *when*.

Every row in this table is a prediction made at a particular month, and the model will be used to predict
the future from the past. A random split - and a grouped one too - happily puts month 17 in training and
month 4 in test, so the model learns from the future in order to predict the past.

Here is what that looks like.

In [ ]:
chronological_side = (stacked.cut_month > 12).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (label, side) in zip(axes, [("random rows", random_side),
                                    ("grouped by member", grouped_side),
                                    ("chronological", chronological_side)]):
    counts = pd.crosstab(stacked.cut_month, side)
    for month in counts.index:
        training = counts.loc[month, 0] if 0 in counts.columns else 0
        testing = counts.loc[month, 1] if 1 in counts.columns else 0
        ax.add_patch(plt.Rectangle((month - 0.42, 0), 0.84, training,
                                   facecolor="#0072B2", edgecolor="white", linewidth=0.6))
        ax.add_patch(plt.Rectangle((month - 0.42, training), 0.84, testing,
                                   facecolor="#D55E00", edgecolor="white", linewidth=0.6))
    if label == "chronological":
        ax.axvline(12.5, color="#000000", linewidth=2)
    ax.set_xlim(0.3, 18.7)
    ax.set_ylim(0, stacked.cut_month.value_counts().max() * 1.12)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("month the prediction is made")
axes[0].set_ylabel("rows")
fig.suptitle("Blue is training, orange is test. Only the third one never learns from the future",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

In the first two panels every month contributes to both sides. **The model is trained on month 17 and
tested on month 4** - which is not a mistake it could make in production, because in production month 17
has not happened.

The middle panel is the important one, because it is the trap. It is a *correctly grouped* split: no
member appears twice, the previous section's leak is fixed, and on the time axis it is indistinguishable
from the random one. Its per-month proportions wobble - whole members land on one side, so some months
get more test rows than others - and that wobble is the only visible difference. **Fixing the grouping
does nothing whatsoever about the chronology.**

The third panel is a **chronological split**: a wall in time, exactly like 04-01's wall, applied to the
splitting rather than to the features.

Now measure all four combinations of the two questions.

In [ ]:
def score_masks(name, train_mask, test_mask):
    train_rows = np.where(train_mask)[0]
    test_rows = np.where(test_mask)[0]
    model = build(name).fit(X.iloc[train_rows], y.iloc[train_rows])
    return roc_auc_score(y.iloc[test_rows], model.predict_proba(X.iloc[test_rows])[:, 1])


early, late = stacked.cut_month <= 12, stacked.cut_month > 12
held_out_members = stacked.member_id % 5 == 0

rows = []
for name in ["logistic regression", "random forest"]:
    random_mean, _ = cross_validate(name, StratifiedKFold(5, shuffle=True, random_state=0))
    grouped_mean, _ = cross_validate(name, GroupKFold(5), groups=groups)
    rows.append({
        "model": name,
        "1. random rows": round(random_mean, 4),
        "2. grouped only": round(grouped_mean, 4),
        "3. chronological only": round(score_masks(name, early, late), 4),
        "4. grouped AND forward": round(score_masks(name, early & ~held_out_members,
                                                    late & held_out_members), 4),
    })
ladder = pd.DataFrame(rows)
print(ladder.to_string(index=False))
print()
print("train rows in setting 4: %d, test rows: %d"
      % ((early & ~held_out_members).sum(), (late & held_out_members).sum()))

In [ ]:
stages = ["1. random rows", "2. grouped only", "3. chronological only", "4. grouped AND forward"]
fig, ax = plt.subplots(figsize=(9.5, 4.8))
for index, record in ladder.iterrows():
    colour = ["#0072B2", "#D55E00"][index]
    values = [record[stage] for stage in stages]
    ax.plot(range(4), values, "o-", color=colour, linewidth=2.2, markersize=9, label=record.model)
    for position, value in enumerate(values):
        ax.text(position, value + 0.006, "%.4f" % value, ha="center", fontsize=8, color=colour)
ax.set_xticks(range(4))
ax.set_xticklabels([s.replace(". ", ".\n") for s in stages], fontsize=9)
ax.set_ylabel("AUC")
ax.set_ylim(0.60, 0.80)
ax.set_title("Each column removes one way of cheating. The gap closes and then reverses", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Read across the two lines, because they tell different stories.

**The forest falls from 0.7580 to 0.6446** - a tenth of an AUC - and most of that (0.0956) is lost at the
first step, when members stop appearing on both sides. Grouping is what the forest was exploiting.

**The logistic regression falls from 0.7313 to 0.6906**, a much smaller drop, and most of *its* loss
comes at the *chronological* step, not the grouping one. It was not memorising members; it was mildly
helped by having seen the whole time range.

**Under the honest split at the right-hand end, the logistic regression wins by 0.046** - and it was
losing by 0.027 at the left-hand end. The crossing is not an artefact of one comparison; it survives
every intermediate stage.

A caution about the fourth column: it trains on 2,740 rows and tests on 560, against 6,318 available. **A
stricter split is also a smaller one**, so part of the fall from column 3 to column 4 is simply less
training data, not more honesty. Separating those two effects is exercise E10, and the habit worth having
is to be suspicious of your own strictest number for exactly this reason.

## Doing it properly: forward chaining

The chronological split above uses one wall, which wastes most of the data - months 13 to 18 never train
anything, and one test period is a sample of size one in time.

**Forward chaining** (also called expanding-window or time-series cross-validation) fixes both: repeat the
wall at several positions, always training on the past and testing on the next block.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
walls = [8, 10, 12, 14, 16]
for fold, wall in enumerate(walls):
    ax.add_patch(plt.Rectangle((0.6, fold - 0.32), wall - 0.1, 0.64,
                               facecolor="#0072B2", edgecolor="white"))
    ax.add_patch(plt.Rectangle((wall + 0.5, fold - 0.32), 2.0, 0.64,
                               facecolor="#D55E00", edgecolor="white"))
    ax.text(wall + 3.0, fold, "train months 1-%d, test %d-%d" % (wall, wall + 1, wall + 2),
            va="center", fontsize=8.5, color="#444444")
ax.set_xlim(0, 27)
ax.set_xticks(range(0, 19, 2))
ax.set_ylim(-0.8, len(walls) - 0.2)
ax.set_yticks(range(len(walls)))
ax.set_yticklabels(["fold %d" % (f + 1) for f in range(len(walls))], fontsize=9)
ax.set_xlabel("month")
ax.set_title("Forward chaining: the training window grows, the test window always comes after",
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for name in ["logistic regression", "random forest"]:
    folds = []
    for wall in walls:
        train_mask = (stacked.cut_month <= wall) & ~held_out_members
        test_mask = (stacked.cut_month > wall) & (stacked.cut_month <= wall + 2) & held_out_members
        folds.append(score_masks(name, train_mask, test_mask))
    rows.append({"model": name, "mean AUC": round(float(np.mean(folds)), 4),
                 "sd across folds": round(float(np.std(folds)), 4),
                 "per fold": " ".join("%.3f" % f for f in folds)})
print(pd.DataFrame(rows).to_string(index=False))

Five estimates instead of one, each honest in both dimensions - the test members are never trained on,
and the test months always come after the training months.

The fold-to-fold spread is the number to report alongside the mean, for the reason 04-03 established: a
single time period is one draw, and time periods differ from each other more than random samples do.

**And notice that this is the first split in the chapter that answers the production question exactly:**
*given everything up to today, how well will this model do next month, on people we have not met?*
Everything else in this chapter answers a slightly different question, and the differences were worth
between 0.02 and 0.10 of AUC.

## Choosing a split: two questions

Everything in this chapter reduces to two questions about the data, asked before any modelling.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.4))

quadrants = [
    (0, 1, "#cfe3f3", "plain random split\n(stratified)",
     "rows are independent\nand order does not matter"),
    (1, 1, "#f6d3bd", "chronological split\nor forward chaining",
     "one row per entity, but\nthe future must not leak"),
    (0, 0, "#cfe8dc", "GroupKFold\n(split by entity)",
     "entities repeat, but there\nis no meaningful time order"),
    (1, 0, "#f3d6e3", "grouped AND forward\n(the strictest)",
     "entities repeat AND\ntime matters - this chapter"),
]
for column, row, colour, title, note in quadrants:
    ax.add_patch(plt.Rectangle((column, row), 0.98, 0.98, facecolor=colour, edgecolor="white",
                               linewidth=3))
    ax.text(column + 0.49, row + 0.72, title, ha="center", va="center", fontsize=12,
            fontweight="bold")
    ax.text(column + 0.49, row + 0.32, note, ha="center", va="center", fontsize=9,
            color="#444444")

ax.set_xlim(-0.02, 2.0)
ax.set_ylim(-0.02, 2.0)
ax.set_xticks([0.49, 1.49])
ax.set_xticklabels(["time does NOT matter", "time DOES matter"], fontsize=11)
ax.set_yticks([0.49, 1.49])
ax.set_yticklabels(["entities\nrepeat", "rows are\nindependent"], fontsize=11)
ax.tick_params(length=0)
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Two questions, four answers", fontsize=13, pad=14)
plt.tight_layout()
plt.show()

**Question 1: does the same entity produce more than one row?** Members, patients, customers, sensors,
documents by the same author, photographs of the same object. If yes, split by that entity.

**Question 2: will the model be used to predict the future from the past?** If yes, the test period must
come after the training period.

Both answers come from the framing (04-01), not from the data science. And both questions have a
mechanical check, which is better than remembering:

```python
assert set(train.member_id) & set(test.member_id) == set()
assert train.cut_month.max() < test.cut_month.min()
```

**Two assertions, in the pipeline, that fail loudly.** Every leak in this chapter would have been caught
by one of them, and neither requires anyone to be paying attention on the day.

## Common misconceptions

**"A random split is the neutral default."**
It is the correct choice for independent rows and a silent bug otherwise. On this data it inflated a
forest by 0.0957 and reversed the model ranking.

**"Cross-validation protects me."**
It repeats whatever split you gave it. Five leaky folds are five leaky estimates, and their small spread
makes the wrong answer look precise: the random-row folds had sd 0.0206 while being 0.096 too high.

**"The problem is that the score is too high."**
The problem is that the score is too high *for some models*. Uniform inflation would be harmless; selective
inflation changes which model wins.

**"I'll group by entity, that covers it."**
Only if time does not matter. Grouping alone left the logistic regression 0.039 above its
grouped-and-forward score.

**"My rows have no obvious group."**
Look harder: near-duplicate documents, the same customer under two ids, photographs from one session, rows
generated by the same upstream job. Grouping is a property of how the data was produced, and it is often
not a column you were given.

**"A stricter split gave a lower score, so it is more honest."**
Partly. It is also usually a *smaller* training set, and some of the drop is that. Compare like with like
before concluding - E10.

**"This only matters for panel data."**
Any repeated measurement: A/B tests with returning users, medical images per patient, sensor readings per
device, rows per household. It is the common case, not the special one.

## Exercises

Solutions: `solutions/04_workflow/04-04_splitting_group_time_solutions.ipynb`.

### Quick understanding

**E1.** State the two questions that determine a splitting strategy, and give the four answers.

**E2.** Why did the random split inflate k-nearest-neighbours far more than logistic regression?

**E3.** In one sentence, why is a reversed model ranking worse than an inflated score?

### Hand calculation

**E4.** A dataset has 1,000 rows from 100 patients, 10 rows each. You split 80/20 at random, row by row.
Compute (a) the probability that a given patient has **no** rows in the training set, and hence the
expected number of genuinely unseen patients, and (b) the probability that a patient has no rows in the
*test* set. (`0.2^10` and `0.8^10`; the second is about 0.107.) Then state, in one sentence, what the two
answers together say about what the test set is measuring.

**E5.** Using this chapter's numbers, compute how much of the forest's total drop (0.7580 to 0.6446) is
attributable to grouping and how much to chronology, using columns 1 to 3 of the ladder. State clearly
why these two do not have to add up to the total.

**E6.** A model is validated with 5-fold cross-validation on a leaky split, giving fold scores 0.755,
0.762, 0.749, 0.771, 0.758. Compute the mean and the standard deviation. Then say what the standard
deviation does and does not tell you about the 0.096 of inflation.

**E7.** Forward chaining with walls at months 8, 10, 12, 14 and 16 and a two-month test window: compute
the total number of *distinct* test months used, and compare with a single chronological split at month
12. Which uses more of the data as test, and which uses more of it for training?

### Coding

**E8.** Write `leak_check(train, test)` that asserts no shared `member_id` and that every training
`cut_month` precedes every test `cut_month`, raising a message that names the violation. Run it on a
random split and on the grouped-and-forward split.

**E9.** Reproduce the inflation table for a decision tree at `max_depth` 2, 5, 10 and unlimited. Plot
inflation against depth. Confirm the claim that inflation grows with capacity, and state where it starts.

**E10.** The fourth ladder column trains on 2,740 rows while the first uses about 5,054. Re-run column 1
with the training set subsampled to 2,740 rows so the comparison is like for like. How much of the drop
was leakage and how much was less data?

**E11.** Implement `forward_chaining_splits(frame, walls, test_window)` yielding index pairs, and use it
with `cross_val_score`-style code to score both models. Report mean and spread.

**E12.** Build the version of this dataset that *would* justify a random split: keep one row per member,
chosen at random. Compare a random split and a grouped split on it, and explain why they now agree.

### Interpretation

**E13.** A colleague's recommender scores 0.94 AUC offline and disappoints in an online test. Rows are
one per user-item interaction. Name the two splits they most likely should have used and what each would
have revealed.

**E14.** Your team's model is retrained monthly and evaluated with a random split. It has looked fine for
a year. Give the argument for changing the evaluation now, and the argument somebody will make against.

### Debugging

**E15.** After switching to `GroupKFold`, your AUC drops from 0.95 to 0.61 and a colleague says the new
code must be broken. Give the three checks that would settle it.

**E16.** A grouped split gives *wildly* different scores per fold - 0.55, 0.81, 0.62, 0.78, 0.59. Give two
explanations and the diagnostic for each.

### Exam and interview reasoning

**E17.** "How would you split this data?" for: (a) 50,000 chest X-rays from 3,000 patients, (b) two years
of daily store sales, (c) 100,000 independent loan applications, (d) 2 million clicks from 40,000 users
over six months. One sentence each, then say which of the four is the trap.

### Transfer to a different situation

**E18.** You are detecting fraud on card transactions. Rows are transactions; cards repeat; fraud arrives
in bursts; and a fraud label is only confirmed 30-60 days after the transaction. Describe your split, and
name the extra problem the labelling delay creates that grouping and chronology do not solve.

### Explain it to someone non-technical

**E19.** Explain in under 90 words why testing a model on new months for existing members is not the same
as testing it on new members, using an everyday analogy.

### Optional challenge

**E20.** The targets in this table *overlap in time*: a prediction made in month 8 asks about months 9-14,
and one made in month 9 asks about months 10-15. Even a grouped-and-forward split leaves this overlap
between the last training rows and the first test rows. Quantify it: build a split with a **gap** of
`HORIZON` months between the training and test periods, and compare with the no-gap version. Does the
score fall, and by how much?

In [ ]:
# Your workspace. In memory: panel, stacked, X, y, groups, FEATURES, frame_at,
# build, cross_validate, score_masks, early, late, held_out_members, walls, ladder.

## Mastery check

- [ ] Detect non-independent rows by counting appearances per entity
- [ ] Draw a random and a grouped split and explain the difference from the picture
- [ ] Predict which models a leaky split will favour, and why
- [ ] Measure inflation by re-splitting rather than by reasoning about it
- [ ] Say why a chronological split is required even when rows are independent
- [ ] Set up forward chaining and report its fold spread
- [ ] Answer the two questions for a dataset you have just been handed
- [ ] Write the two assertions that would have caught every leak in this chapter

## What should now feel instinctive

- Counting rows per entity before choosing a splitting function
- Reaching for `GroupKFold` whenever an id column repeats
- Asking "does the model predict forward in time?" and splitting on time when it does
- Distrusting a cross-validated score whose folds agree suspiciously well
- Putting the two assertions in the pipeline rather than in your memory

## Flashcards

| Front | Back |
|---|---|
| When a random split is wrong | Rows are not independent, or time matters |
| Grouped split | Every row from one entity is on the same side. `GroupKFold` |
| Chronological split | Test period comes after the training period. No exceptions |
| Forward chaining | Repeat the chronological wall at several positions; the training window grows |
| Overlap here | 539 of 544 test members (99.1%) were also in training |
| Inflation, logistic regression | +0.0021. Seven coefficients cannot memorise a member |
| Inflation, random forest | +0.0957 |
| Inflation, k-nearest neighbours | +0.1052. kNN *is* memory; it looks the member up |
| The rule about inflation | It scales with a model's capacity to memorise, so it is not uniform |
| The real damage | The ranking reverses: random says forest, honest says logistic regression |
| Why CV does not save you | It repeats the split you gave it; five leaky folds agree closely and are all wrong |
| The two assertions | No shared entity ids; max training time < min test time |
| Strictest is also smallest | Part of any drop is less training data. Subsample to compare fairly |

## Next

**04-05 · Leakage lab: target, temporal, duplicate, preprocessing.** This chapter covered two of the four
ways the answer gets into the features - **duplicate** leakage, where the same entity is on both sides,
and **temporal** leakage, where the future is. 04-01 already showed the third, **target** leakage, with
`months_on_file`.

The fourth is the one nobody sees coming, because it happens in code that looks like tidying up:
fitting a scaler, an imputer or an encoder on the whole dataset before splitting. The next chapter puts
all four in one lab, with a diagnostic for each - and then 04-06 and 04-07 build the pipeline machinery
that makes the fourth impossible rather than merely discouraged.